# Prosody-Syntax Distributional Analysis

This notebook builds a co-occurrence matrix of (lemma, POS) tags against a mix of
**syntactic** and **prosodic** features extracted from the SUD-French-Rhapsodie
treebank. From that matrix we derive three parallel views — syntactic-only,
prosodic-only, and combined — and use each to compute:

1. **Similarity scores** between lexical units
2. A **per-unit purity / entropy / rival-category** score
3. A **category-level transition matrix** (how categories bleed into each other)
4. **Bridge words** — units whose neighbourhood pulls them toward a different POS
5. **Pulling features** — the features responsible for that pull
6. A global **weighted-average stability** for each feature space: syntactic, prosodic or both

The key cleanup vs. the original notebook: every downstream analysis takes a
`space ∈ {"syntactic", "prosodic", "combined"}` selector, so swapping feature
spaces is a one-line change.

In [ ]:
import sys
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

from scipy.stats import entropy
from sklearn.metrics.pairwise import cosine_similarity

# ── Paths ──────────────────────────────────────────────────────────────────
TOD_PATH      = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/tod"
GREX_PATH     = "/Users/madalina/Documents/M2TAL/stage/grex/grex2"
TREEBANK_PATH = "/Users/madalina/Documents/PHD/code/data/SUD_French-Rhapsodie/prosody_pauses"
PATTERNS_FILE = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/scripts/3. probability_matrix/patterns_all_nodes_prosody.txt"

GREW_PATTERN      = "pattern{X[upos<>PUNCT]}"
ANALYSED_CATEGORY = "all_nodes"   # kept for reference; not used downstream

# ── Bring third-party packages onto the path ───────────────────────────────
for p in (TOD_PATH, GREX_PATH):
    if p not in sys.path:
        sys.path.insert(1, p)

import pyximport
pyximport.install()

import grewpy
from grewpy import Corpus, CorpusDraft, Request

import tod.corpus  # noqa: F401  (registers handlers; keep import)

import grex.data           # noqa: F401
import grex.utils
import grex.features       # noqa: F401
import grex.data_prosody

grewpy.set_config("sud")


In [ ]:
# ── Drop entirely: redundant, unreliable, or too fine-grained ─────────────
FEATURES_TO_DROP = {
    'MeanF0',           # use SemitonesFromUtteranceMean instead (speaker-normalised)
    'MeanF0Step',       # threshold artefact, not a prosodic observation
    'AvgAmplitude',     # recording-condition-dependent
    'MaxAmplitude',     # same
    'SylForm',          # phonetic transcription, too fine-grained
    'AlignBegin',       # timing offset, not linguistic
    'AlignEnd',         # timing offset, not linguistic
    'pauseID',          # identifier, not a feature
    # Before*/Next* equivalents of dropped features:
    'BeforeMeanF0', 'BeforeMeanF0Step', 'BeforeAvgAmplitude',
    'NextMeanF0',   'NextMeanF0Step',   'NextAvgAmplitude',
    'Semiton', 'Overlap', 'Loc', 'Glo', 'Backchannel',
}

# ── Bin into 3 quantile-based categories ──────────────────────────────────
# Note: Duration appears in TWO units in the data:
#   - syllable subnodes: integer ms     (e.g. 289, 167)
#   - pause nodes:       float seconds  (e.g. 0.2695, 0.3328)
# We normalise to ms before binning — see `_normalise_duration` below.
FEATURES_TO_BIN = {
    'SemitonesFromUtteranceMean',
    'Duration',
    'BeforeSemitonesFromUtterance',
    'NextSemitonesFromUtterance',
    'BeforeDuration',
    'NextDuration',
    'BeforeSemiton',
    'NextSemiton',
    'NextSemitonsFromUtterance',
}

BIN_LABELS = {
    'SemitonesFromUtteranceMean':   ['F0_low',       'F0_mid',       'F0_high'],
    'Duration':                     ['Dur_short',    'Dur_mid',      'Dur_long'],
    'BeforeSemitonesFromUtterance': ['BeforeF0_low', 'BeforeF0_mid', 'BeforeF0_high'],
    'NextSemitonesFromUtterance':   ['NextF0_low',   'NextF0_mid',   'NextF0_high'],
}


In [ ]:
# ── SLAM3 contour decomposition ───────────────────────────────────────────
# The *Tone features (FootTone, GroupTone, PackageTone, Period_tone) encode
# pitch contours compositionally:
#
#     [start_pitch][end_pitch]            -- no salient peak (2 chars)
#     [start_pitch][end_pitch][peak][tier] -- with salient peak (4 chars)
#
# pitch ∈ {L, l, m, h, H}  (5 ordinal levels: very low → very high)
# tier  ∈ {1, 2, 3}        (which syllabic tier carries the peak)
# Plus the special atomic value 'Pause'.
#
# Treating these as opaque strings gives ~300 sparse one-hot columns per
# *Tone feature. Decomposing into {start, end, peak, tier, has_peak,
# direction, magnitude} yields ~5 dense sub-features per token instead.

SLAM_TONE_FEATURES = {'FootTone', 'GroupTone', 'PackageTone', 'Period_tone'}

# Ordinal mapping used to derive direction & magnitude.
_PITCH_LEVELS = {'L': 1, 'l': 2, 'm': 3, 'h': 4, 'H': 5}
_TIERS = {'1', '2', '3'}


def _parse_slam(label):
    """Parse a SLAM3 contour string into its components.

    Returns a dict {pause, start, end, peak, tier} or None if the label
    doesn't match the SLAM3 grammar (silently skipped downstream).
    """
    if label == 'Pause':
        return {'pause': True, 'start': None, 'end': None,
                'peak': None, 'tier': None}

    if (len(label) == 2
            and label[0] in _PITCH_LEVELS and label[1] in _PITCH_LEVELS):
        return {'pause': False, 'start': label[0], 'end': label[1],
                'peak': None, 'tier': None}

    if (len(label) == 4
            and label[0] in _PITCH_LEVELS and label[1] in _PITCH_LEVELS
            and label[2] in _PITCH_LEVELS and label[3] in _TIERS):
        return {'pause': False, 'start': label[0], 'end': label[1],
                'peak': label[2], 'tier': label[3]}

    return None


def _slam_subfeatures(prefix, label):
    """Yield 'sub-feature=value' strings derived from a SLAM3 contour label.

    `prefix` is the path up to and including the *Tone name,
    e.g. 'node:X:child:GroupTone'. Sub-features become
    'node:X:child:GroupTone:start=H', 'node:X:child:GroupTone:direction=rise', …
    """
    parsed = _parse_slam(label)
    if parsed is None:
        return

    if parsed['pause']:
        yield f"{prefix}:pause=True"
        return

    s, e = parsed['start'], parsed['end']
    yield f"{prefix}:start={s}"
    yield f"{prefix}:end={e}"

    # Direction & magnitude on the ordinal pitch scale.
    delta = _PITCH_LEVELS[e] - _PITCH_LEVELS[s]
    if delta > 0:
        direction = 'rise'
    elif delta < 0:
        direction = 'fall'
    else:
        direction = 'flat'
    yield f"{prefix}:direction={direction}"
    yield f"{prefix}:magnitude={abs(delta)}"

    if parsed['peak'] is not None:
        yield f"{prefix}:has_peak=True"
        yield f"{prefix}:peak={parsed['peak']}"
        yield f"{prefix}:tier={parsed['tier']}"
    else:
        yield f"{prefix}:has_peak=False"

In [ ]:
def _normalise_duration(value_str):
    """Convert a Duration string to milliseconds regardless of source node type.

    Syllable subnodes store ms (e.g. '289'); pause nodes store seconds
    (e.g. '0.2695'). Heuristic: values < 10 are seconds. A literal 0.0 means
    'missing' (e.g. overlap nodes) → returns None.
    """
    try:
        v = float(value_str)
    except (ValueError, TypeError):
        return None
    if v == 0.0:
        return None
    return v * 1000 if v < 10 else v


def _is_sentinel(value_str):
    """True for the '0.0' sentinel that overlap / missing pause nodes carry.

    Without this guard, those zeros would be binned as F0_low rather than
    silently dropped.
    """
    try:
        return float(value_str) == 0.0
    except (ValueError, TypeError):
        return False


def _feature_name(key):
    """Bare feature name from a path-like key, e.g. ('node','X','child','Duration') → 'Duration'."""
    return key[-1]


In [ ]:
def collect_continuous_values(all_data_items):
    """Walk every (lex_unit, match, features) triple and collect raw float
    values for each feature in FEATURES_TO_BIN.

    Returns: {feature_name: [float, ...]}
    """
    collector = {f: [] for f in FEATURES_TO_BIN}

    for _, _, features in all_data_items:
        for key, value in features.items():
            fname = _feature_name(key)
            if fname not in FEATURES_TO_BIN:
                continue
            # `value` is a string for own/parent/prev/next, or a set of strings
            # for child (a node can have multiple syllables).
            values = value if isinstance(value, set) else {value}
            for v in values:
                if fname == 'Duration':
                    norm = _normalise_duration(v)
                    if norm is not None:
                        collector['Duration'].append(norm)
                elif not _is_sentinel(v):
                    try:
                        collector[fname].append(float(v))
                    except (ValueError, TypeError):
                        pass
    return collector


def compute_thresholds(collector, n_bins=3):
    """Quantile boundaries (interior only) per feature.

    Returns: {feature_name: [boundary_1, boundary_2, ...]}
    """
    thresholds = {}
    for fname, values in collector.items():
        if not values:
            continue
        arr = np.array(values)
        quantiles = np.linspace(0, 1, n_bins + 1)[1:-1]
        thresholds[fname] = np.quantile(arr, quantiles).tolist()
    return thresholds


In [ ]:
def _bin_value(fname, value_str, thresholds):
    """Continuous string → bin label. Returns None for sentinels / unknowns."""
    if fname not in thresholds or _is_sentinel(value_str):
        return None

    if fname == 'Duration':
        v = _normalise_duration(value_str)
    else:
        try:
            v = float(value_str)
        except (ValueError, TypeError):
            return None

    if v is None:
        return None

    bounds = thresholds[fname]
    labels = BIN_LABELS.get(fname, [f'{fname}_Q1', f'{fname}_Q2', f'{fname}_Q3'])
    for i, b in enumerate(bounds):
        if v <= b:
            return labels[i]
    return labels[-1]


def format_features(features, thresholds):
    """Turn a feature dict into a list of 'k1:k2:...:kn=value' strings.

    Continuous features are binned; dropped features are skipped.
    """
    result = []
    for key, value in features.items():
        fname = _feature_name(key)
        prefix = ':'.join(key)

        if fname in FEATURES_TO_DROP:
            continue

        if fname in FEATURES_TO_BIN:
            values = value if isinstance(value, set) else {value}
            for v in values:
                label = _bin_value(fname, v, thresholds)
                if label is not None:
                    result.append(f"{prefix}={label}")
            continue

        # SLAM3 contour features: decompose into structural sub-features.
        if fname in SLAM_TONE_FEATURES:
            values = value if isinstance(value, set) else {value}
            for v in values:
                result.extend(_slam_subfeatures(prefix, v))
            continue


        # Categorical: keep as-is.
        if isinstance(value, set):
            for v in value:
                result.append(f"{prefix}={v}")
        else:
            result.append(f"{prefix}={value}")

    return result


def _filter_features(features, excluded_patterns=None, included_patterns=None):
    """Regex-based include/exclude filter over a list of feature strings."""
    import re
    out = []
    for feature in features:
        if excluded_patterns and any(re.search(p, feature) for p in excluded_patterns):
            continue
        if included_patterns and not any(re.search(p, feature) for p in included_patterns):
            continue
        out.append(feature)
    return out


In [ ]:
corpus = grewpy.Corpus(TREEBANK_PATH)
draft  = grewpy.CorpusDraft(corpus)

all_matches = corpus.search(
    grewpy.Request(GREW_PATTERN)
        .without("X[InIdiom=Yes]")
        .without("X[Idiom=Yes]")
        .without("X[InTitle=Yes]")
        .without("X[Title=Yes]")
        .without("X[Scrap=Yes]")
        .without("X[Foreign]")
        .without("X[Lang]")
        .without("X-[fixed]->Y")
        .without("Y-[flat:name]->X")
        .without("Y-[goeswith]->X"),
    clustering_parameter=["X.lemma"],
)

# Drop lemmas with fewer than 10 occurrences.
matches = {k: v for k, v in all_matches.items() if len(v) > 10}

# sent_id → {node_id: features}
sent_id_to_sentence = {
    draft[i].meta["sent_id"]: draft[i].features for i in range(len(draft))  # type: ignore
}

print(f"Lemmas after frequency filter: {len(matches)}")


In [ ]:
# Re-key by (lemma, POS). Use ExtPos when present (e.g. '%' has upos=SYM but
# ExtPos=NOUN, and we want NOUN). Otherwise fall back to upos.
match_upos = {}
for lemma, occurrences in matches.items():
    for m in occurrences:
        sent_id = m["sent_id"]
        node_id = str(m["matching"]["nodes"]["X"])
        sentence_features = sent_id_to_sentence.get(sent_id)
        if sentence_features is None or node_id not in sentence_features:
            continue
        token_features = sentence_features[node_id]
        if "ExtPos" in token_features:
            pos = token_features["ExtPos"]
        elif "upos" in token_features:
            pos = token_features["upos"]
        else:
            print(f"WARNING: lemma '{lemma}' in '{sent_id}' has no upos.")
            continue
        match_upos.setdefault((lemma, pos), []).append(m)

# Filter again post-split: a lemma may have passed at the lemma level but
# split into rare (lemma, POS) pairs.
match_upos = {k: v for k, v in match_upos.items() if len(v) > 10}

print(f"(lemma, POS) units after frequency filter: {len(match_upos)}")


In [ ]:
# Load the feature template config and run feature extraction over every match.
with open(PATTERNS_FILE) as f:
    config = yaml.load(f, Loader=yaml.Loader)

templates = grex.utils.FeaturePredicate.from_config(config["templates"])
feature_predicate = grex.utils.FeaturePredicate.from_config(
    config["features"], templates=templates,
)

all_data_items = []
for lex_unit, mts in match_upos.items():
    for match in mts:
        feats = grex.data_prosody.extract_features(draft, match, feature_predicate)
        all_data_items.append((lex_unit, match, feats))

print(f"Total (lex_unit, match, features) triples: {len(all_data_items)}")


In [ ]:
# Compute corpus-wide quantile thresholds, then format every match's features
# (binning the continuous ones, dropping the unwanted ones).
collector  = collect_continuous_values(all_data_items)
thresholds = compute_thresholds(collector, n_bins=3)

print("Computed thresholds:")
for fname, bounds in thresholds.items():
    labels = BIN_LABELS.get(fname, ['Q1', 'Q2', 'Q3'])
    print(
        f"  {fname}: <{bounds[0]:.3f} → {labels[0]} | "
        f"<{bounds[1]:.3f} → {labels[1]} | ≥{bounds[1]:.3f} → {labels[2]}"
    )

# Drop 'own' features (the focused node's own attributes — they trivially
# determine its category and would dominate similarity).
EXCLUDED_FEATURE_PATTERNS = ['own', 'ExtPos', "Loc=", "direction=flat", "has_peak=False", "FootTone:pause", "GroupTone:pause", "PackageTone:pause", "Period_tone:pause", "ProminenceFinal=Pause", "ProminenceInitial=Pause"]

data = {k: [] for k in match_upos}
for lex_unit, _, features in all_data_items:
    formatted = _filter_features(
        format_features(features, thresholds),
        excluded_patterns=EXCLUDED_FEATURE_PATTERNS,
    )
    data[lex_unit].append(formatted)


In [ ]:
# Build index lookups. These are used everywhere downstream.
unique_lemma    = sorted(set(data))
unique_features = sorted({feat for lists in data.values() for fs in lists for feat in fs})

idx2feature = {i: feat for i, feat in enumerate(unique_features)}
feature2idx = {feat: i for i, feat in idx2feature.items()}
idx2lexunit = {i: feat for i, feat in enumerate(unique_lemma)}
lexunit2idx = {feat: i for i, feat in idx2lexunit.items()}

print(f"Lexical units: {len(idx2lexunit)} | features: {len(idx2feature)}")


In [ ]:
idx2feature

In [ ]:
co_occurrence_matrix = np.zeros((len(idx2lexunit), len(idx2feature)))
for lex_unit, feature_lists in data.items():
    for feature_list in feature_lists:
        for feature in feature_list:
            co_occurrence_matrix[lexunit2idx[lex_unit], feature2idx[feature]] += 1

row_sums = co_occurrence_matrix.sum(axis=1)
coverage_matrix = np.divide(
    co_occurrence_matrix,
    row_sums[:, np.newaxis],
    out=np.zeros_like(co_occurrence_matrix),
    where=row_sums[:, np.newaxis] != 0,
)

feature_matrix = coverage_matrix


In [ ]:
# Heuristic: prosodic features are those whose name contains any of these
# substrings. Everything else is syntactic.
PROSODIC_MARKERS = (
    'childSyl', 'prevSyl', 'nextSyl',
    'Slope', 'Glo', 'Loc', 'Duration',
    'PitchRange', 'AvgHeight', 'Semiton',
    'F0', 'Fused', 'Foot', 'Group', 'Package',  # (substring match also catches FootTone/FootType, GroupTone/RhythmGroup, PackageTone/Package_type, Period_tone, ProminenceFinal/Initial, …)
    'Period', 'Prominence', 'Hesitation',
    'RhythmGroup', 'NextBreakLength'
)

prosodic_mask = np.array([
    any(p in feat for p in PROSODIC_MARKERS)
    for feat in idx2feature.values()
])
syntactic_mask = ~prosodic_mask

print(f"Syntactic features: {syntactic_mask.sum()}")
print(f"Prosodic  features: {prosodic_mask.sum()}")

# Cosine similarities for each space. `combined` uses the full matrix.
similarities = {
    "syntactic": cosine_similarity(feature_matrix[:, syntactic_mask]),
    "prosodic":  cosine_similarity(feature_matrix[:, prosodic_mask]),
    "combined":  cosine_similarity(feature_matrix),
}


In [ ]:
# Global category frequencies — denominator of P_norm.
all_pos = [lu[1] for lu in idx2lexunit.values()]
total_pos_count = len(all_pos)
category_freqs = {cat: count / total_pos_count for cat, count in Counter(all_pos).items()}


def compute_scores(similarity, idx2lexunit, category_freqs,
                   sim_threshold=0.7, max_neighbours=20):
    """For every lexical unit, return purity / entropy / rival-category metrics
    based on the given similarity matrix.

    Returns: {(lemma, POS): {"purity", "entropy", "rival_category", "rival_weight_ratio"}}
    """
    scores = {}

    for idx, lex_unit in idx2lexunit.items():
        current_category = lex_unit[1]

        # 1. Top-k neighbours above similarity threshold (excluding self).
        candidate_idx = np.where(similarity[idx] >= sim_threshold)[0]
        neighbours = sorted(
            [i for i in candidate_idx if i != idx],
            key=lambda i: similarity[idx][i],
            reverse=True,
        )[:max_neighbours]

        if not neighbours:
            continue

        # 2. Sum similarity weight per category.
        category_weights = {}
        purity_bottom = 0.0
        for n_idx in neighbours:
            n_cat = idx2lexunit[n_idx][1]
            w = similarity[idx][n_idx]
            purity_bottom += w
            category_weights[n_cat] = category_weights.get(n_cat, 0) + w

        # 3. P_norm = (raw share) / (global frequency), then renormalise to sum to 1.
        norm_weights = {
            cat: (w / purity_bottom) / category_freqs[cat]
            for cat, w in category_weights.items()
        }
        total = sum(norm_weights.values())
        p_norm = {cat: v / total for cat, v in norm_weights.items()}

        # 4. Purity & entropy.
        purity_score  = p_norm.get(current_category, 0.0)
        entropy_score = entropy(list(p_norm.values()))

        # 5. Strongest non-self category.
        rival_cat = None
        rival_p   = 0.0
        for cat, p in p_norm.items():
            if cat != current_category and p > rival_p:
                rival_p   = p
                rival_cat = cat

        scores[lex_unit] = {
            "purity":             purity_score,
            "entropy":            entropy_score,
            "rival_category":     rival_cat,
            "rival_weight_ratio": rival_p,
        }

    return scores


In [ ]:
# Compute scores for all three feature spaces in one pass. Pick whichever you
# want for downstream analysis.
scores_by_space = {
    space: compute_scores(sim, idx2lexunit, category_freqs)
    for space, sim in similarities.items()
}

print({space: len(s) for space, s in scores_by_space.items()})


In [ ]:
def build_transition_matrix(similarity, idx2lexunit, category_freqs,
                            sim_threshold=0.7, max_neighbours=20):
    """Square row-stochastic DataFrame indexed by POS category."""
    rows = []

    for idx, lex_unit in idx2lexunit.items():
        current_cat = lex_unit[1]

        candidate_idx = np.where(similarity[idx] > sim_threshold)[0]
        neighbours = sorted(
            [i for i in candidate_idx if i != idx],
            key=lambda i: similarity[idx][i],
            reverse=True,
        )[:max_neighbours]

        if not neighbours:
            continue

        raw_weights = {}
        total_raw_weight = 0.0
        for n_idx in neighbours:
            n_cat = idx2lexunit[n_idx][1]
            sim = similarity[idx][n_idx]
            raw_weights[n_cat] = raw_weights.get(n_cat, 0) + sim
            total_raw_weight += sim

        norm_probs = {
            cat: (w / total_raw_weight) / category_freqs[cat]
            for cat, w in raw_weights.items()
        }
        total = sum(norm_probs.values())
        for cat in norm_probs:
            norm_probs[cat] /= total
            rows.append({"Original": current_cat, "Neighbor": cat,
                         "Weight":   norm_probs[cat]})

    df = pd.DataFrame(rows)

    # Force the matrix to be square — add any missing rows/columns as zero.
    all_categories = sorted(set(df['Original']) | set(df['Neighbor']))
    matrix = (df.groupby(['Original', 'Neighbor'])['Weight']
                .sum()
                .unstack(fill_value=0)
                .reindex(index=all_categories, columns=all_categories,
                         fill_value=0))

    # Row-normalise (rows that summed to 0 stay 0).
    row_sums = matrix.sum(axis=1).replace(0, 1)
    matrix_to_return = matrix.div(row_sums, axis=0)
    matrix_to_return = matrix_to_return.drop(index='X', columns='X', errors='ignore')
    return matrix_to_return


def plot_transition_matrix(matrix, title, drop_categories=()):
    """Heatmap of a transition matrix. `drop_categories` removes rows + cols
    (useful for hiding 'X' / 'SYM' which are usually noise)."""
    if drop_categories:
        keep = [c for c in matrix.index if c not in drop_categories]
        matrix = matrix.loc[keep, keep]
    plt.figure(figsize=(12, 10))
    sns.heatmap(matrix, annot=True, cmap="YlOrRd", fmt=".2f")
    plt.title(title)
    plt.ylabel("Original Category (from Treebank)")
    plt.xlabel("Rival Category (The 'Intruder')")
    plt.show()


In [ ]:
# Build a transition matrix for each feature space.
transition_matrices = {
    space: build_transition_matrix(sim, idx2lexunit, category_freqs)
    for space, sim in similarities.items()
}


# Show one — change SPACE to flip view.
SPACE = "prosodic"   # "syntactic" | "prosodic" | "combined"
plot_transition_matrix(
    transition_matrices[SPACE],
    title=f"Stability Matrix — Rhapsodie ({SPACE})",
    # drop_categories=("X", "SYM"),   # uncomment to hide noise categories
)
transition_matrices[SPACE]


In [ ]:
def get_top_bridge_words(scores_dict, original_cat, rival_cat, top_n=100):
    """Words from `original_cat` most pulled toward `rival_cat`."""
    matches = [
        {"word": word, "purity": float(d["purity"]), "entropy": float(d["entropy"])}
        for (word, o_cat), d in scores_dict.items()
        if o_cat == original_cat and d["rival_category"] == rival_cat
    ]
    # Lowest combined score first — least pure, highest entropy.
    matches.sort(key=lambda x: 0.7 * x["purity"] - 0.3 * x["entropy"])
    return [(m["word"], f'{m["purity"]:.2f}', f'{m["entropy"]:.2f}')
            for m in matches[:top_n]]


In [ ]:
# ── Pick a feature space and a category pair ──────────────────────────────
SPACE          = "prosodic"        # "syntactic" | "prosodic" | "combined"
ORIGINAL_CAT   = "ADJ"
RIVAL_CAT      = "ADV"

scores = scores_by_space[SPACE]
bridge_words = get_top_bridge_words(scores, ORIGINAL_CAT, RIVAL_CAT)

print(f"Top {ORIGINAL_CAT} → {RIVAL_CAT} bridge words ({SPACE} space):")
print(f"{'Word':<20} {'Purity':>8} {'Entropy':>8}")
for word, purity, ent in bridge_words:
    print(f"{word:<20} {purity:>8} {ent:>8}")


In [ ]:
def get_pulling_features(
    lex_unit, rival_category,
    feature_matrix, similarity_matrix,
    idx2lexunit, lexunit2idx,
    idx2feature, feature2idx,
    top_k_features=20,
    sim_threshold=0.7, max_neighbours=20,
):
    """Features pulling `lex_unit` toward `rival_category`.

    Score = (unit's coverage on the feature) / (|unit_value − rival_centroid| + ε).
    Features the unit doesn't use at all (zero coverage) score zero — they
    can't be pulling anything.
    """
    unit_idx = lexunit2idx[lex_unit]
    unit_vector = feature_matrix[unit_idx]

    # Top-k neighbours.
    candidate_idx = np.where(similarity_matrix[unit_idx] >= sim_threshold)[0]
    neighbour_indices = sorted(
        [i for i in candidate_idx if i != unit_idx],
        key=lambda i: similarity_matrix[unit_idx][i],
        reverse=True,
    )[:max_neighbours]

    # Rival centroid: median over rival-category neighbours.
    rival_neighbour_indices = [
        i for i in neighbour_indices if idx2lexunit[i][1] == rival_category
    ]
    if not rival_neighbour_indices:
        print(f"No neighbours of category '{rival_category}' found for {lex_unit}.")
        return []
    rival_centroid = np.median(feature_matrix[rival_neighbour_indices], axis=0)

    feature_differences = np.abs(unit_vector - rival_centroid)
    eps = 1e-6
    scores = np.where(
        unit_vector > 0,
        unit_vector / (feature_differences + eps),
        0.0,
    )

    top_idx = np.argsort(scores)[::-1][:top_k_features]
    return [
        {
            "feature":    idx2feature[f_idx],
            "unit_value": unit_vector[f_idx],
            "rival_avg":  rival_centroid[f_idx],
            "diff":       feature_differences[f_idx],
            "score":      scores[f_idx],
        }
        for f_idx in top_idx
        if scores[f_idx] > 0
    ]


In [ ]:
def get_aggregate_pulling_features(
    scores_dict, original_cat, rival_cat,
    feature_matrix, similarity_matrix,
    idx2lexunit, lexunit2idx,
    idx2feature, feature2idx,
    top_k_words=10, top_k_features=20,
    sim_threshold=0.7, max_neighbours=20,
):
    """Cross-word aggregate of the pulling features for an `original_cat`
    → `rival_cat` direction.

    A feature appears in the aggregate if it surfaced for at least one bridge
    word. Ranked by how many bridge words it shows up in (robustness), with
    average unit value as a tiebreaker.
    """
    bridge_words = get_top_bridge_words(scores_dict, original_cat, rival_cat,
                                        top_n=top_k_words)
    if not bridge_words:
        print(f"No bridge words found from {original_cat} to {rival_cat}")
        return None

    print(f"Analyzing {len(bridge_words)} bridge words: "
          f"{[w[0] for w in bridge_words]}\n")

    per_feature = {}
    for word, _purity, _entropy in bridge_words:
        lex_unit = (word, original_cat)
        if lex_unit not in lexunit2idx:
            print(f"  WARNING: {lex_unit} not found in lexunit2idx, skipping.")
            continue

        word_features = get_pulling_features(
            lex_unit=lex_unit, rival_category=rival_cat,
            feature_matrix=feature_matrix, similarity_matrix=similarity_matrix,
            idx2lexunit=idx2lexunit, lexunit2idx=lexunit2idx,
            idx2feature=idx2feature, feature2idx=feature2idx,
            top_k_features=top_k_features * 2,   # cast wide before aggregating
            sim_threshold=sim_threshold, max_neighbours=max_neighbours,
        )
        for fd in word_features:
            d = per_feature.setdefault(fd["feature"],
                                       {"unit_values": [], "rival_avgs": [], "diffs": []})
            d["unit_values"].append(fd["unit_value"])
            d["rival_avgs"].append(fd["rival_avg"])
            d["diffs"].append(fd["diff"])

    if not per_feature:
        print("No features collected — check bridge words and similarity threshold.")
        return None

    rows = [
        {
            "feature":         fname,
            "count_words":     len(d["unit_values"]),
            "avg_unit_value":  float(np.mean(d["unit_values"])),
            "avg_rival_value": float(np.mean(d["rival_avgs"])),
            "avg_diff":        float(np.mean(d["diffs"])),
            "std_unit_value":  float(np.std(d["unit_values"])),
        }
        for fname, d in per_feature.items()
    ]
    rows.sort(key=lambda r: (r["count_words"], r["avg_unit_value"]), reverse=True)
    return pd.DataFrame(rows[:top_k_features])


In [ ]:
# ── Single-word usage ──────────────────────────────────────────────────────
SPACE        = "prosodic"          # "syntactic" | "prosodic" | "combined"
LEX_UNIT     = ("magique", "ADJ")
RIVAL_CAT    = "ADV"

results = get_pulling_features(
    LEX_UNIT, RIVAL_CAT,
    feature_matrix, similarities[SPACE],
    idx2lexunit, lexunit2idx,
    idx2feature, feature2idx,
    top_k_features=20,
)

print(f"Top features pulling {LEX_UNIT} toward {RIVAL_CAT} ({SPACE} space):")
print(f"{'Feature':<55} {'unit':>6}  {'rival':>6}  {'diff':>6}  {'score':>8}")
print("-" * 90)
for r in results:
    print(
        f"{r['feature']:<55} "
        f"{r['unit_value']:>6.3f}  "
        f"{r['rival_avg']:>6.3f}  "
        f"{r['diff']:>6.3f}  "
        f"{r['score']:>8.2f}"
    )


In [ ]:
# ── Aggregate usage (the "defining features" of a directional pull) ───────
SPACE        = "prosodic"           # "syntactic" | "prosodic" | "combined"
ORIGINAL_CAT = "NOUN"
RIVAL_CAT    = "VERB"

df_agg = get_aggregate_pulling_features(
    scores_dict=scores_by_space[SPACE],
    original_cat=ORIGINAL_CAT, rival_cat=RIVAL_CAT,
    feature_matrix=feature_matrix,
    similarity_matrix=similarities[SPACE],
    idx2lexunit=idx2lexunit, lexunit2idx=lexunit2idx,
    idx2feature=idx2feature, feature2idx=feature2idx,
    top_k_words=5, top_k_features=20,
)

if df_agg is not None:
    print(df_agg.to_string(index=False))


In [ ]:
def calculate_weighted_stability(transition_matrix, category_counts):
    """Weighted average of the diagonal of the transition matrix.

    transition_matrix : square, row-stochastic, indexed by POS.
    category_counts   : dict / Series mapping POS → token count (or any
                         consistent weight).
    """
    categories = transition_matrix.index
    stability_scores = pd.Series(np.diag(transition_matrix), index=categories)
    weights = pd.Series(category_counts).reindex(categories).fillna(0)
    total_weight = weights.sum()
    if total_weight == 0:
        return 0.0
    return (stability_scores * weights).sum() / total_weight


# WAS for every feature space.
for space, tm in transition_matrices.items():
    was = calculate_weighted_stability(tm, category_freqs)
    print(f"WAS ({space:<9}): {was:.4f}")


In [ ]:
def permutation_null(similarity, idx2lexunit, n_permutations=200, seed=0):
    rng = np.random.default_rng(seed)
    pos_labels = np.array([lu[1] for lu in idx2lexunit.values()])
    lemmas     = [lu[0] for lu in idx2lexunit.values()]

    # category_freqs depends only on the count of each label, which is
    # preserved under permutation, so we can compute it once.
    cat_counts = Counter(pos_labels)
    n_total = len(pos_labels)
    cat_freqs = {c: k / n_total for c, k in cat_counts.items()}

    null_matrices = []
    for _ in range(n_permutations):
        shuffled = rng.permutation(pos_labels)
        idx2lex_perm = {i: (lemmas[i], shuffled[i]) for i in range(len(lemmas))}
        m = build_transition_matrix(similarity, idx2lex_perm, cat_freqs)
        null_matrices.append(m)

    # Align all matrices on the same category set
    all_cats = sorted(set().union(*[m.index for m in null_matrices]))
    aligned  = [m.reindex(index=all_cats, columns=all_cats, fill_value=0)
                for m in null_matrices]
    stack    = np.stack([m.values for m in aligned])      # shape (n_perm, k, k)
    return (pd.DataFrame(stack.mean(axis=0), index=all_cats, columns=all_cats),
            pd.DataFrame(stack.std(axis=0),  index=all_cats, columns=all_cats))

null_mean, null_std = permutation_null(similarities["prosodic"], idx2lexunit)
observed = transition_matrices["prosodic"]
z_scores = (observed - null_mean) / null_std.replace(0, np.nan)

In [ ]:
print("Z-scores for prosodic transition matrix vs. permutation null:")
print(z_scores)

In [ ]:
plot_transition_matrix(
    z_scores,
    title="Z-scores vs. Permutation Null — Rhapsodie (Prosodic Space)",
    drop_categories=("X", "SYM"),   # hide noise categories
)